# PyTorch - Layer Types

Explore and research different PyTorch layer types.

# Notebook Setup

## Imports

In [11]:
# Import Standard Libraries
import torch
import torch.functional as F

# Layer Types

## Linear Layer

In [10]:
# Check linear layer parameters
linear = torch.nn.Linear(3, 2)

for index, param in enumerate(linear.parameters()):
    # Switch between weights and bias
    parameter_name = 'Weights' if index == 0 else 'Bias'
    print('Parameter: ', parameter_name)
    print('Shape: ', param.shape)
    print(param)
    print()

# Check weights
print('Linear Layer weights -', linear.weight)

Parameter:  Weights
Shape:  torch.Size([2, 3])
Parameter containing:
tensor([[-0.1807, -0.4746,  0.2120],
        [ 0.1208,  0.4870, -0.4575]], requires_grad=True)

Parameter:  Bias
Shape:  torch.Size([2])
Parameter containing:
tensor([-0.0179, -0.3414], requires_grad=True)

Linear Layer weights - Parameter containing:
tensor([[-0.1807, -0.4746,  0.2120],
        [ 0.1208,  0.4870, -0.4575]], requires_grad=True)


## Convolutional Layer

In [12]:
class LeNet(torch.nn.Module):
    """1x32x32 black and white images"""

    def __init__(self):
        super(LeNet, self).__init__()
        # 1 input image channel (black & white), 6 output channels (number of features we want to compute), 5x5 square convolution
        # kernel
        self.conv1 = torch.nn.Conv2d(1, 6, 5) # Output "Activation Map": 6 x 28 x 28
        self.conv2 = torch.nn.Conv2d(6, 16, 3)
        # an affine operation: y = Wx + b
        self.fc1 = torch.nn.Linear(16 * 6 * 6, 120)  # 6*6 from image dimension
        self.fc2 = torch.nn.Linear(120, 84)
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        # Convolutional -> Relu -> Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

## Recurrent Layers

In [13]:
class LSTMTagger(torch.nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        """

        Args:
            embedding_dim: Size of the embedding space -> i.e., how big is each embedding vector
            hidden_dim: Size of LSTM memory
            vocab_size: Number of words in the vocabulary
            tagset_size: Number of tags in the outputset
        """
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = torch.nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

# Data Manipulation Layers

## Max Pooling Layer

It reduce the tensor dimension by combining cells together and assigning the maimum value of those cells.

In [14]:
my_tensor = torch.rand(1, 6, 6)
print(my_tensor)

maxpool_layer = torch.nn.MaxPool2d(3)
print(maxpool_layer(my_tensor))

tensor([[[0.9564, 0.2641, 0.2794, 0.8561, 0.7306, 0.4079],
         [0.7903, 0.8390, 0.0909, 0.5783, 0.4731, 0.0779],
         [0.4989, 0.5272, 0.4833, 0.4338, 0.3632, 0.1433],
         [0.7672, 0.0725, 0.0242, 0.6033, 0.0121, 0.5534],
         [0.4525, 0.3907, 0.4771, 0.7569, 0.6996, 0.2587],
         [0.2034, 0.3804, 0.7859, 0.4251, 0.7494, 0.2934]]])
tensor([[[0.9564, 0.8561],
         [0.7859, 0.7569]]])


## Normalisation Layer

It re-center the and normalise the output of one layer before feeding it to another.

Notice how the mean becomes close to zero.

In [15]:
my_tensor = torch.rand(1, 4, 4) * 20 + 5
print(my_tensor)

print(my_tensor.mean())

norm_layer = torch.nn.BatchNorm1d(4)
normed_tensor = norm_layer(my_tensor)
print(normed_tensor)

print(normed_tensor.mean())

tensor([[[22.2820, 14.7284, 12.8306, 22.9240],
         [22.6388, 14.8853,  6.6825, 16.9133],
         [11.1946, 18.8417, 12.5825, 11.3445],
         [ 7.5648, 17.8198, 11.2145,  5.3320]]])
tensor(14.3612)
tensor([[[ 0.9155, -0.7750, -1.1997,  1.0592],
         [ 1.2864, -0.0690, -1.5030,  0.2855],
         [-0.7322,  1.7063, -0.2897, -0.6844],
         [-0.6172,  1.5519,  0.1548, -1.0894]]],
       grad_fn=<NativeBatchNormBackward0>)
tensor(5.2154e-08, grad_fn=<MeanBackward0>)
